# H(p,q) workflow
This notebook demonstrates the analysis workflow using the pre-defined functions in `src/compute_H.py`.
It does not redefine any of the library functions -- it imports and reuses them -- and displays all generated images and saved grids from the `outputs/` directory.

In [ ]:
# Import existing functions and helpers
from compute_H import *
import numpy as np
import os
import matplotlib.pyplot as plt
from IPython.display import display, Image
# Ensure inline plotting works when running interactively
%matplotlib inline

## Fast: generate analytic p$\\to$0 spectra 

In [ ]:
# Generate analytic spectra for a set of Mach numbers and save to outputs/
M_values = [0.001, 0.01, 0.1, 1.0]
out_dir = os.path.abspath('outputs')
os.makedirs(out_dir, exist_ok=True)
out_png = os.path.join(out_dir, 'H_spectra_analytic.png')

# Call the function - it will append M and R values (2-digit scientific notation) to the filename
plot_spectra_M_analytic(M_values, qmin=1e-4, qmax=1e1, nq=300, out_png=out_png, R=1e4)

# Construct the actual filename with M and R appended (matching 2-digit scientific notation format)
mlist_str = '-'.join([f"{M:.2e}".replace('+', 'p').replace('-', 'm') for M in M_values])
rstr = f"{1e4:.2e}".replace('+', 'p').replace('-', 'm')
out_png_base, out_png_ext = os.path.splitext(out_png)
actual_png = f"{out_png_base}_Ms{mlist_str}_R{rstr}{out_png_ext}"

if os.path.exists(actual_png):
    display(Image(filename=actual_png))
else:
    print(f"File not found: {actual_png}")


## Optional (slow): full 2D H(p,q) scan
The full 2D scan is expensive. Uncomment and run if you want the grid generation and full p--q image.

In [ ]:
# Example: run a slower 2D scan (can take a while!)
# To enable, uncomment the example_scan_and_plot call below
print("2D scan cell: Disabled by default (slow computation).")
print("To run the full 2D scan, uncomment the lines below.")

# out_png = os.path.join(out_dir, 'H_pq.png')
# out_npy = os.path.join(out_dir, 'Hgrid.npy')
# 
# # Run the 2D scan with M=1.0 and R=1e4
# # This will generate output files with M and R appended to the name
# print("Running 2D scan... this may take several minutes")
# example_scan_and_plot(out_png=out_png, out_npy=out_npy, M=1.0, R=1e4)
# 
# # The function appends _M1e+00_R1e+04 to filenames (2-digit scientific notation):
# # - H_pq_M1e+00_R1e+04.png
# # - Hgrid_M1e+00_R1e+04.npy
# 
# mstr = f"{1.0:.2e}".replace('+', 'p').replace('-', 'm')
# rstr = f"{1e4:.2e}".replace('+', 'p').replace('-', 'm')
# out_png_with_params = f"{out_png.replace('.png', '')}_M{mstr}_R{rstr}.png"
# 
# if os.path.exists(out_png_with_params):
#     print("Display 2D scan result:")
#     display(Image(filename=out_png_with_params))


## Display all PNG images in `outputs/`
This will show all existing PNGs found in the repository `outputs/` directory.

In [ ]:
# Display every PNG in outputs/
out_dir = os.path.abspath('outputs')
pngs = sorted([f for f in os.listdir(out_dir) if f.lower().endswith('.png')])
print(f'Found {len(pngs)} PNG files in', out_dir)
for fn in pngs:
    path = os.path.join(out_dir, fn)
    print(' ---', fn, '---')
    display(Image(filename=path))

## Visualize saved H grids (.npy)
If the `.npy` files were saved as dictionaries with keys `ps`, `qs`, and `H`, this cell will load and plot them.

In [ ]:
# Load and visualize any Hgrid .npy files in outputs/
npy_files = sorted([f for f in os.listdir(out_dir) if f.lower().endswith('.npy')])
for fn in npy_files:
    path = os.path.join(out_dir, fn)
    print(' ---', fn, '---')
    try:
        data = np.load(path, allow_pickle=True)
        # many of the saved files are saved as a dict-like object
        if isinstance(data, np.ndarray) and data.dtype == object:
            data = data.item()
        if isinstance(data, dict) and set(['ps','qs','H']).issubset(data.keys()):
            ps = data['ps']
            qs = data['qs']
            H = data['H']
        else:
            # Fallback: try expected structure [ps, qs, H] or raw array
            if hasattr(data, 'item'):
                try:
                    d = data.item()
                    if isinstance(d, dict) and set(['ps','qs','H']).issubset(d.keys()):
                        ps = d['ps']; qs = d['qs']; H = d['H']
                    else:
                        print('Unknown npy format for', fn); continue
                except Exception:
                    print('Cannot interpret', fn); continue
            else:
                print('Cannot interpret', fn); continue
        # Plot H grid
        plt.figure(figsize=(6,4))
        plt.pcolormesh(ps, qs, H, shading='auto', cmap='viridis')
        plt.xscale('log'); plt.yscale('log')
        plt.xlabel('p = k/k0'); plt.ylabel(r'q = $\\omega$/k0')
        plt.title(fn)
        plt.colorbar(label='H')
        plt.tight_layout()
        display(plt.gcf())
        plt.close()
    except Exception as e:
        print('Error loading', fn, e)

---
### Notes and next steps
- The analytic spectra generation uses `plot_spectra_M_analytic` (fast).
- The full 2D scan `example_scan_and_plot` is provided but left commented because it can be very slow.
- If you want, I can run the optional 2D scan cell, or adjust plotting styles / export formats.